In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# ──────────────────────────────────────────────────────────────
# Global figure style
# ──────────────────────────────────────────────────────────────


mpl.rcParams.update({
    'font.family':        'serif',
    'font.serif':         ['Nimbus Roman', 'DejaVu Serif', 'Liberation Serif'],
    'font.size':          11,
    'axes.linewidth':     0.8,
    'axes.edgecolor':     'black',
    'xtick.direction':    'in',
    'ytick.direction':    'in',
    'xtick.major.size':   4.5,
    'ytick.major.size':   4.5,
    'xtick.minor.size':   2.5,
    'ytick.minor.size':   2.5,
    'xtick.top':          False,   # spine visible, but no ticks on top
    'ytick.right':        False,   # spine visible, but no ticks on right
    'figure.dpi':         130,
    'savefig.dpi':        300,
    'axes.spines.top':    True,    # keep the box frame complete
    'axes.spines.right':  True,
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

In [ ]:
transform    = transforms.ToTensor()
trainset     = torchvision.datasets.MNIST(root='./data', train=True,
                                           download=True, transform=transform)
testset      = torchvision.datasets.MNIST(root='./data', train=False,
                                           download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=256, shuffle=True)
test_loader  = torch.utils.data.DataLoader(testset,  batch_size=256, shuffle=False)

class SimpleCNN(nn.Module):
    """
    Two convolutional layers followed by two fully-connected layers.
    Input:  1 x 28 x 28 grayscale image
    Output: 10 class scores (one per digit)
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)   # 32 filters, 3x3 kernel
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)  # 64 filters
        self.fc1   = nn.Linear(64*7*7, 128)            # 128 hidden units
        self.fc2   = nn.Linear(128, 10)                # 10 output classes
    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))    # -> 32 x 14 x 14
        x = F.relu(F.max_pool2d(self.conv2(x), 2))    # -> 64 x 7 x 7
        x = x.view(-1, 64*7*7)                         # flatten
        return self.fc2(F.relu(self.fc1(x)))           # -> 10 scores

criterion = nn.CrossEntropyLoss()
print('Model defined. Parameters:',
      sum(p.numel() for p in SimpleCNN().parameters()), )

In [ ]:
model     = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epoch_losses = []   # store average loss of each epoch
num_epochs = 20 # Train for 3 epochs，which should be enough to reach ~98% accuracy on MNIST, we use 20 epochs to show a nice loss curve
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg = running_loss / len(train_loader)
    epoch_losses.append(avg)
    print(f'Epoch {epoch+1}/{num_epochs}  |  avg loss: {avg:.4f}')

# Evaluate on the test set
model.eval()
correct = 0
all_preds = []
all_labels = []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1)

        correct += (preds == lbls).sum().item()
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(lbls.cpu().tolist())
clean_acc = correct / len(testset)
print(f'\nClean test accuracy: {clean_acc*100:.2f}%')

# ──────────────────────────────────────────────────────────────
# Visualization 1: Loss curve
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    range(1, num_epochs + 1),
    epoch_losses,
    color='#2166AC',
    marker='o',
    linewidth=1.8,
    markersize=5,
    markerfacecolor='white',
    markeredgewidth=1.6,
    markeredgecolor='#2166AC',
    zorder=3,
    label='Training Loss',
)

# ── Dynamically detect convergence epoch ───────────────────────
drops      = [epoch_losses[i-1] - epoch_losses[i] for i in range(1, len(epoch_losses))]
threshold  = max(drops) * 0.06
conv_idx   = next((i for i, d in enumerate(drops) if d < threshold), 5)
conv_epoch = conv_idx + 2              # 1-indexed; epoch *after* the steep descent
conv_loss  = epoch_losses[conv_epoch - 1]
loss_range = max(epoch_losses) - min(epoch_losses)

# Shaded region: rapid descent phase
ax.axvspan(0.3, conv_epoch, alpha=0.07, color='#2166AC', zorder=1)
ax.text(
    (1 + conv_epoch) / 2,
    min(epoch_losses) + loss_range * 0.70,
    'Rapid\nDescent',
    ha='center', va='center', fontsize=8.5,
    color='#2166AC', style='italic', alpha=0.85,
)

# Arrow + annotation box at the convergence point
ax.annotate(
    f'Loss stabilizes\n(around epoch {conv_epoch})',
    xy=(conv_epoch, conv_loss),
    xytext=(conv_epoch + 2.6, conv_loss + loss_range * 0.30),
    fontsize=9.5,
    ha='left', va='center',
    arrowprops=dict(
        arrowstyle='->',
        color='#444444',
        lw=1.1,
        connectionstyle='arc3,rad=-0.25',
    ),
    bbox=dict(
        boxstyle='round,pad=0.5',
        facecolor='#F0F5FF',
        edgecolor='#8899BB',
        linewidth=0.8,
        alpha=0.92,
    ),
)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Average Training Loss', fontsize=12)
ax.set_title('Training Loss Curve on MNIST', fontsize=13, pad=9)
ax.set_xlim(0.3, num_epochs + 0.7)
ax.set_xticks(range(1, num_epochs + 1, 2))
ax.grid(True, linestyle='--', linewidth=0.45, alpha=0.45, color='gray')
ax.legend(fontsize=10, framealpha=0.7, edgecolor='#aaaaaa')

plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────────────────────
# Visualization 2: Sample test predictions
# ──────────────────────────────────────────────────────────────
imgs, lbls = next(iter(test_loader))
imgs, lbls = imgs[:8].to(device), lbls[:8].to(device)

with torch.no_grad():
    outputs = model(imgs)
    preds = outputs.argmax(1)

fig, axes = plt.subplots(1, 8, figsize=(11, 2.0))
fig.subplots_adjust(top=0.72, bottom=0.02, left=0.01, right=0.99, wspace=0.12)

for i, ax in enumerate(axes):
    correct_pred = preds[i].item() == lbls[i].item()
    border_color = '#2166AC' if correct_pred else '#D6001C'
    ax.imshow(imgs[i].cpu().squeeze(), cmap='gray', interpolation='nearest')
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(2.0)
    ax.set_title(
        f'Pred: {preds[i].item()}\nTrue: {lbls[i].item()}',
        fontsize=9,
        color=border_color,
        pad=4,
    )
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(
    'Sample Test Predictions  (blue = correct, red = wrong)',
    fontsize=11,
    y=0.98,
)
plt.show()


In [ ]:
def fgsm_attack(image, epsilon, grad):
    """
    Apply one FGSM step.
    Args:
        image   : original input tensor (N, C, H, W)
        epsilon : perturbation budget -- how far any pixel can move
        grad    : gradient of the loss w.r.t. the image pixels
    Returns:
        adversarial image, pixel values clamped to [0, 1]
    """
    sign_grad = grad.sign()           # +1 or -1 for each pixel
    x_adv     = image + epsilon * sign_grad
    return torch.clamp(x_adv, 0, 1)  # keep pixel values valid

In [ ]:
def evaluate_fgsm(model, loader, epsilon, n_batches=4):
    """
    Run FGSM on up to n_batches and return:
      - accuracy on adversarial examples
      - a list of (original, adversarial, true_label, orig_pred, adv_pred)
        tuples for images where the prediction changed
    """
    model.eval()
    correct, total, examples = 0, 0, []

    for batch_idx, (images, labels) in enumerate(loader):
        if batch_idx >= n_batches:
            break
        images, labels = images.to(device), labels.to(device)

        # Step 1: enable gradient tracking on the INPUT (not the weights)
        images.requires_grad = True

        # Step 2: forward pass -- compute model output and loss
        outputs = model(images)
        loss    = criterion(outputs, labels)

        # Step 3: backward pass -- compute gradient of loss w.r.t. pixels
        model.zero_grad()
        loss.backward()
        # images.grad now contains the gradient for each pixel

        # Step 4: apply FGSM using the pixel gradient
        adv_images = fgsm_attack(images, epsilon, images.grad.detach())

        # Step 5: re-evaluate the model on the adversarial images
        with torch.no_grad():
            adv_outputs = model(adv_images)

        correct += (adv_outputs.argmax(1) == labels).sum().item()
        total   += labels.size(0)

        # Collect a few examples where the prediction changed
        if len(examples) < 5:
            orig_pred = outputs.argmax(1)
            adv_pred  = adv_outputs.argmax(1)
            for i in range(len(labels)):
                if orig_pred[i] != adv_pred[i] and len(examples) < 5:
                    examples.append({
                        'orig':      images[i].detach().cpu(),
                        'adv':       adv_images[i].detach().cpu(),
                        'true':      labels[i].item(),
                        'orig_pred': orig_pred[i].item(),
                        'adv_pred':  adv_pred[i].item(),
                    })
    return correct / total, examples



In [ ]:
def evaluate_fgsm_on_batch(model, images, labels, epsilon):
    """
    Run FGSM on a single batch and return:
      - accuracy on adversarial examples
      - adversarial images
      - original predictions
      - adversarial predictions
    """
    model.eval()

    images = images.clone().detach().to(device)
    labels = labels.to(device)
    images.requires_grad_(True)

    outputs = model(images)
    loss = criterion(outputs, labels)

    model.zero_grad()
    loss.backward()

    adv_images = fgsm_attack(images, epsilon, images.grad.detach())

    with torch.no_grad():
        adv_outputs = model(adv_images)

    adv_acc = (adv_outputs.argmax(1) == labels).float().mean().item()

    return (
        adv_acc,
        adv_images.detach(),
        outputs.argmax(1).detach(),
        adv_outputs.argmax(1).detach(),
    )

In [ ]:
# ══════════════════════════════════════════════════════════════
# Visualization 3 & 4: FGSM Attack Analysis
# ══════════════════════════════════════════════════════════════

epsilons = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

accs      = []
all_exs   = {}   # epsilon -> list of example dicts
for eps in epsilons:
    acc, exs = evaluate_fgsm(model, test_loader, eps, n_batches=8)
    accs.append(acc * 100)
    all_exs[eps] = exs
    print(f'  ε = {eps:.2f}  →  adv accuracy: {acc*100:.1f}%')

# ──────────────────────────────────────────────────────────────
# Visualization 3: Accuracy vs ε line plot
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6.5, 3.8))

ax.plot(
    epsilons, accs,
    color='#D6001C',
    marker='o',
    linewidth=1.8,
    markersize=5.5,
    markerfacecolor='white',
    markeredgewidth=1.6,
    markeredgecolor='#D6001C',
    zorder=3,
    label='Adversarial accuracy',
)

# Clean-accuracy reference line (ε = 0)
ax.axhline(
    y=accs[0], color='#2166AC', linewidth=1.2,
    linestyle='--', alpha=0.7, label=f'Clean accuracy ({accs[0]:.1f}%)',
)

# Annotate the drop at ε = 0.3
drop = accs[0] - accs[-1]
ax.annotate(
    f'−{drop:.1f}% accuracy\nat ε = {epsilons[-1]}',
    xy=(epsilons[-1], accs[-1]),
    xytext=(epsilons[-1] - 0.12, accs[-1] + 12),
    fontsize=9.5,
    ha='center', va='bottom',
    arrowprops=dict(
        arrowstyle='->',
        color='#444444',
        lw=1.0,
        connectionstyle='arc3,rad=0.2',
    ),
    bbox=dict(
        boxstyle='round,pad=0.45',
        facecolor='#FFF0F0',
        edgecolor='#CC7777',
        linewidth=0.75,
        alpha=0.92,
    ),
)

ax.set_xlabel('Perturbation Budget \u03b5', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('FGSM Attack: Accuracy vs. Perturbation Budget', fontsize=13, pad=9)
ax.set_xlim(-0.01, 0.32)
ax.set_ylim(0, 105)
ax.set_xticks(epsilons)
ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%g%%'))
ax.grid(True, linestyle='--', linewidth=0.45, alpha=0.45, color='gray')
ax.legend(fontsize=10, framealpha=0.75, edgecolor='#aaaaaa', loc='upper right')

plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────────────────────
# Visualization 4: Adversarial examples gallery
#   Layout: 3 rows × N_cols
#     Row 0 – Original image
#     Row 1 – Perturbation (ε · sign(∇)) × 10 for visibility
#     Row 2 – Adversarial image
# ──────────────────────────────────────────────────────────────
eps_show  = 0.20                        # which epsilon to display
examples  = all_exs[eps_show][:5]       # up to 5 misclassified examples
N         = len(examples)

ROW_LABELS = [
    'Original\n(clean)',
    f'Perturbation\n(×10, ε = {eps_show})',
    'Adversarial\n(attacked)',
]
ROW_COLORS = ['#2166AC', '#888800', '#D6001C']

fig, axes = plt.subplots(3, N, figsize=(2.1 * N, 6.0))
fig.subplots_adjust(left=0.14, right=0.98, top=0.88, bottom=0.04,
                    hspace=0.08, wspace=0.06)

for col, ex in enumerate(examples):
    orig  = ex['orig'].squeeze().numpy()          # H x W
    adv   = ex['adv'].squeeze().numpy()
    pert  = np.clip((adv - orig) * 10 + 0.5, 0, 1)   # centered, ×10 magnified

    rows_data = [orig, pert, adv]
    rows_cmap = ['gray', 'RdBu_r', 'gray']

    for row in range(3):
        ax = axes[row, col]
        ax.imshow(rows_data[row], cmap=rows_cmap[row],
                  vmin=0, vmax=1, interpolation='nearest')

        # Colored frame per row
        for spine in ax.spines.values():
            spine.set_edgecolor(ROW_COLORS[row])
            spine.set_linewidth(1.6)
        ax.set_xticks([])
        ax.set_yticks([])

        # Column header: pred/true labels (top row only)
        if row == 0:
            ax.set_title(
                f'True: {ex["true"]}',
                fontsize=9, color='#333333', pad=4,
            )
        # Bottom row: show orig pred → adv pred
        if row == 2:
            ax.set_xlabel(
                f'{ex["orig_pred"]} \u2192 {ex["adv_pred"]}',
                fontsize=10, color='#D6001C', labelpad=3,
            )

# Row labels on the left side
for row, (label, color) in enumerate(zip(ROW_LABELS, ROW_COLORS)):
    axes[row, 0].set_ylabel(
        label, fontsize=9, color=color,
        rotation=90, labelpad=6, va='center',
    )

fig.suptitle(
    f'FGSM Adversarial Examples (ε = {eps_show}) — red arrows show fool label (orig → adv)',
    fontsize=11,
    y=0.94,
    x=0.55,
    ha='center'
)
plt.show()

In [ ]:
def pgd_attack(model, images, labels, epsilon=0.1, alpha=0.025, iters=40):
    """
    Projected Gradient Descent attack.
    Args:
        epsilon : total L-inf perturbation budget
        alpha   : step size per iteration (typically epsilon/4)
        iters   : number of gradient steps (typically 40)
    """
    model.eval()

    # Start from a random point inside the epsilon-ball
    # (random init avoids getting stuck in poor local optima)
    noise     = torch.zeros_like(images).uniform_(-epsilon, epsilon)
    perturbed = torch.clamp(images + noise, 0, 1).detach()

    for step in range(iters):
        perturbed.requires_grad_(True)

        # One FGSM-style step
        loss = criterion(model(perturbed), labels)
        model.zero_grad()
        loss.backward()

        with torch.no_grad():
            # Move in the sign-gradient direction
            perturbed = perturbed + alpha * perturbed.grad.sign()

            # Project: clip so no pixel moves more than epsilon from original
            delta     = torch.clamp(perturbed - images, -epsilon, epsilon)
            perturbed = torch.clamp(images + delta, 0, 1)

    return perturbed.detach()

In [ ]:
imgs, lbls = next(iter(test_loader))
imgs, lbls = imgs[:256].to(device), lbls[:256].to(device)

model.eval()
with torch.no_grad():
    clean_preds = model(imgs).argmax(1)
    clean_acc   = (clean_preds == lbls).float().mean().item()

fgsm_adv_preds = evaluate_fgsm_on_batch(
    model, imgs, lbls, epsilon=0.1
)

pgd_adv = pgd_attack(model, imgs, lbls, epsilon=0.1)
with torch.no_grad():
    pgd_preds = model(pgd_adv).argmax(1)
    pgd_acc   = (pgd_preds == lbls).float().mean().item()

# Text summary
print(f'{"Condition":<22} {"Epsilon":>8} {"Accuracy":>10}')
print('-' * 44)
print(f'{"No attack (clean)":<22} {"---":>8} {clean_acc*100:>9.1f}%')
print(f'{"FGSM (1 step)":<22} {0.1:>8.2f} {fgsm_acc*100:>9.1f}%')
print(f'{"PGD (40 steps)":<22} {0.1:>8.2f} {pgd_acc*100:>9.1f}%')


In [ ]:
# ══════════════════════════════════════════════════════════════
# Visualization 5: Accuracy bar chart — Clean vs FGSM vs PGD
# ══════════════════════════════════════════════════════════════
conditions = ['Clean\n(no attack)', 'FGSM\n(1 step)', 'PGD\n(40 steps)']
acc_vals   = [clean_acc * 100, fgsm_acc * 100, pgd_acc * 100]
bar_colors = ['#2166AC', '#E08214', '#D6001C']

fig, ax = plt.subplots(figsize=(5.5, 3.8))
bars = ax.bar(conditions, acc_vals, color=bar_colors, width=0.52,
              edgecolor='black', linewidth=0.6, zorder=3)
for bar, acc in zip(bars, acc_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=10.5,
            fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Accuracy Under Different Attacks  (\u03b5 = 0.1)',
             fontsize=13, pad=9)
ax.set_ylim(0, 112)
ax.grid(True, axis='y', linestyle='--', linewidth=0.45, alpha=0.45, color='gray')
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# Visualization 6: PGD adversarial examples gallery
# ══════════════════════════════════════════════════════════════
fooled = (clean_preds != pgd_preds).nonzero(as_tuple=True)[0][:5]
N_gal  = len(fooled)

ROW_LABELS = ['Original\n(clean)', 'Perturbation\n(\u00d710)', 'Adversarial\n(PGD-40)']
ROW_COLORS = ['#2166AC', '#888800', '#D6001C']

fig, axes = plt.subplots(3, N_gal, figsize=(2.1 * N_gal, 6.0))
fig.subplots_adjust(left=0.14, right=0.98, top=0.88, bottom=0.04,
                    hspace=0.08, wspace=0.06)
for col, idx in enumerate(fooled):
    i = idx.item()
    orig = imgs[i].cpu().squeeze().numpy()
    adv  = pgd_adv[i].cpu().squeeze().numpy()
    pert = np.clip((adv - orig) * 10 + 0.5, 0, 1)
    for row, (data, cmap) in enumerate(zip(
            [orig, pert, adv], ['gray', 'RdBu_r', 'gray'])):
        ax = axes[row, col]
        ax.imshow(data, cmap=cmap, vmin=0, vmax=1, interpolation='nearest')
        for sp in ax.spines.values():
            sp.set_edgecolor(ROW_COLORS[row]); sp.set_linewidth(1.6)
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            ax.set_title(f'True: {lbls[i].item()}', fontsize=9,
                         color='#333', pad=4)
        if row == 2:
            ax.set_xlabel(
                f'{clean_preds[i].item()} \u2192 {pgd_preds[i].item()}',
                fontsize=10, color='#D6001C', labelpad=3)
for row, (lbl, clr) in enumerate(zip(ROW_LABELS, ROW_COLORS)):
    axes[row, 0].set_ylabel(lbl, fontsize=9, color=clr,
                            rotation=90, labelpad=6, va='center')
fig.suptitle('PGD Adversarial Examples (\u03b5 = 0.1, 40 steps)',
             fontsize=11, y=0.94)
plt.show()

# ══════════════════════════════════════════════════════════════
# Prepare data for FGSM vs PGD animation (next cell)
# ══════════════════════════════════════════════════════════════

def pgd_with_snapshots(model, image, label, eps=0.1, alpha=0.025, iters=40):
    """Return list[dict] with image / probs / pred / conf / loss at every step."""
    model.eval()
    snaps = []
    noise = torch.zeros_like(image).uniform_(-eps, eps)
    x = torch.clamp(image + noise, 0, 1).detach()
    for t in range(iters + 1):
        if t > 0:
            x.requires_grad_(True)
            loss = criterion(model(x), label)
            model.zero_grad(); loss.backward()
            with torch.no_grad():
                x = x + alpha * x.grad.sign()
                x = torch.clamp(image + torch.clamp(x - image, -eps, eps), 0, 1)
        with torch.no_grad():
            out   = model(x)
            probs = F.softmax(out, dim=1)
        snaps.append(dict(
            image=x.detach().cpu().squeeze().numpy(),
            probs=probs.cpu().squeeze().numpy(),
            pred=out.argmax(1).item(),
            conf=probs.max().item(),
            loss=criterion(out, label).item(),
        ))
    return snaps

# Pick a demo image — prefer visually complex digits (not 1/0)
preferred = [4, 7, 8, 9, 5, 3, 2, 6]
demo_i = fooled[0].item()
for idx in fooled:
    if lbls[idx.item()].item() in preferred:
        demo_i = idx.item()
        break

demo_img = imgs[demo_i:demo_i+1]
demo_lbl = lbls[demo_i:demo_i+1]
true_cls = lbls[demo_i].item()
orig_np  = demo_img.cpu().squeeze().numpy()

# FGSM on demo image
d = demo_img.detach().clone().requires_grad_(True)
model.zero_grad()
criterion(model(d), demo_lbl).backward()
fgsm_demo = fgsm_attack(d, 0.1, d.grad.detach())
with torch.no_grad():
    fo = model(fgsm_demo); fp = F.softmax(fo, dim=1)
    fgsm_pred = fo.argmax(1).item()
    fgsm_conf = fp.max().item()
with torch.no_grad():
    oo = model(demo_img); op = F.softmax(oo, dim=1)
    orig_pred = oo.argmax(1).item()
    orig_conf = op.max().item()

# PGD snapshots (step 0..40)
pgd_snaps = pgd_with_snapshots(model, demo_img.detach(), demo_lbl,
                                eps=0.1, iters=40)
# FGSM "snapshots" (step 0 = original, step 1 = done)
fgsm_snaps = [
    dict(image=orig_np, pred=orig_pred, conf=orig_conf),
    dict(image=fgsm_demo.detach().cpu().squeeze().numpy(),
         pred=fgsm_pred, conf=fgsm_conf),
]

print(f'\nDemo image: true label = {true_cls}')
print(f'  FGSM: {orig_pred} -> {fgsm_pred}  (conf {fgsm_conf*100:.0f}%)')
print(f'  PGD:  {orig_pred} -> {pgd_snaps[-1]["pred"]}  (conf {pgd_snaps[-1]["conf"]*100:.0f}%)')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Visualization 7: FGSM vs PGD — Side-by-Side Animation
# ══════════════════════════════════════════════════════════════
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as IPImage, display

N_FRAMES = 41

fig, axes = plt.subplots(2, 2, figsize=(8, 6.5),
                          gridspec_kw={'height_ratios': [1.3, 1]})
fig.subplots_adjust(left=0.05, right=0.95, top=0.78, bottom=0.08,
                    hspace=0.35, wspace=0.20)

# ── Persistent overlay for explanatory annotations ──────────
ax_ov = fig.add_axes([0, 0, 1, 1], facecolor='none')
ax_ov.set_xlim(0, 1); ax_ov.set_ylim(0, 1); ax_ov.axis('off')

def animate(frame):
    fgsm_step = min(frame, 1)
    fs = fgsm_snaps[fgsm_step]
    ps = pgd_snaps[frame]

    for row in axes:
        for ax in row:
            ax.clear()

    # Clear and redraw overlay
    ax_ov.clear()
    ax_ov.set_xlim(0, 1); ax_ov.set_ylim(0, 1); ax_ov.axis('off')

    # ── Row 0: Current images ─────────────────────────────────
    for col, (s, method, step, total, mclr) in enumerate([
        (fs, 'FGSM', fgsm_step, 1, '#E08214'),
        (ps, 'PGD',  frame,    40, '#D6001C'),
    ]):
        ax = axes[0, col]
        ax.imshow(s['image'], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        pclr = '#D6001C' if s['pred'] != true_cls else '#2166AC'
        for sp in ax.spines.values(): sp.set_edgecolor(pclr); sp.set_linewidth(2.5)
        ax.set_xticks([]); ax.set_yticks([])

        done = '  ✔ done!' if method == 'FGSM' and step == 1 else ''
        ax.set_title(f'{method}  —  Step {step} / {total}{done}',
                     fontsize=14, color=mclr, fontweight='bold', pad=8)
        ax.set_xlabel(f'Pred: {s["pred"]}   Conf: {s["conf"]*100:.0f}%',
                      fontsize=12, color=pclr, fontweight='bold', labelpad=5)

    # ── Row 1: Perturbation (diff from original) ─────────────
    for col, (s, mclr, label) in enumerate([
        (fs, '#E08214', 'FGSM Perturbation'),
        (ps, '#D6001C', 'PGD Perturbation'),
    ]):
        ax = axes[1, col]
        pert = s['image'] - orig_np
        ax.imshow(pert, cmap='RdBu_r', vmin=-0.12, vmax=0.12, interpolation='nearest')
        for sp in ax.spines.values(): sp.set_edgecolor(mclr); sp.set_linewidth(1.8)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(label, fontsize=12, color=mclr, pad=6)
        linf = np.abs(pert).max()
        l2   = np.sqrt((pert ** 2).sum())
        ax.set_xlabel(f'L∞ = {linf:.3f}    L₂ = {l2:.2f}',
                      fontsize=11, color='#555', labelpad=4)

    # ── Suptitle ──────────────────────────────────────────────
    fig.suptitle(
        f'FGSM vs PGD: Attack Process  (ε = 0.1,  True label = {true_cls})',
        fontsize=15, y=0.95, fontweight='bold',
    )

    # ── Explanatory annotations on the overlay ────────────────

    # Top band between suptitle and plots: FGSM explanation (left)
    ax_ov.text(0.26, 0.87,
        'FGSM: compute gradient once,\n'
        'perturb by ε·sign(∇L) in one shot.',
        fontsize=9, color='#E08214', linespacing=1.4,
        va='center', ha='center',
        bbox=dict(boxstyle='round,pad=0.35', fc='#FFF8EE',
                  ec='#E08214', lw=0.8, alpha=0.92))

    # Top band: PGD explanation (right)
    ax_ov.text(0.74, 0.87,
        'PGD: many small FGSM-like steps,\n'
        'projecting back to ε-ball each time.',
        fontsize=9, color='#D6001C', linespacing=1.4,
        va='center', ha='center',
        bbox=dict(boxstyle='round,pad=0.35', fc='#FFF5F5',
                  ec='#D6001C', lw=0.8, alpha=0.92))

    # Dynamic commentary — positioned in the gap between row 0 and row 1
    if frame == 0:
        note = '▶ Both start from the same clean image'
        note_clr = '#2166AC'
    elif frame == 1:
        note = '▶ FGSM finishes in 1 step — PGD has just begun'
        note_clr = '#E08214'
    elif frame <= 10:
        note = f'▶ PGD step {frame}/40 — gradually refining perturbation...'
        note_clr = '#D6001C'
    elif frame <= 20:
        pgd_conf = ps['conf'] * 100
        note = f'▶ PGD step {frame}/40 — model confidence shifting ({pgd_conf:.0f}%)'
        note_clr = '#D6001C'
    elif frame <= 35:
        note = f'▶ PGD step {frame}/40 — perturbation pattern converging'
        note_clr = '#D6001C'
    else:
        note = f'▶ PGD step {frame}/40 — attack nearly complete'
        note_clr = '#D6001C'

    ax_ov.text(0.50, 0.39,
        note, fontsize=10.5, color=note_clr,
        ha='center', va='center', style='italic',
        bbox=dict(boxstyle='round,pad=0.3', fc='white',
                  ec=note_clr, lw=0.8, alpha=0.85))

    # Bottom: perturbation explanation
    ax_ov.text(0.50, 0.025,
        'Perturbation maps:  Red = pixel increased  |  Blue = pixel decreased  |  '
        'L∞ = max change (bounded by ε)',
        fontsize=8.5, color='#666', ha='center', va='bottom',
        bbox=dict(boxstyle='round,pad=0.3', fc='#F8F8F8',
                  ec='#CCC', lw=0.6, alpha=0.90))

    return []

anim = FuncAnimation(fig, animate, frames=N_FRAMES, interval=250, blit=False)
gif_path = 'fgsm_vs_pgd_process.gif'
anim.save(gif_path, writer=PillowWriter(fps=5))
plt.close(fig)

print(f'Animation saved to: {gif_path}')
display(IPImage(filename=gif_path))

# ══════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════
# Visualization 8: FGSM vs PGD — Step-by-Step Breakdown
# ══════════════════════════════════════════════════════════════
key_steps = [0, 5, 10, 20, 30, 40]

fig, axes = plt.subplots(3, 6, figsize=(15, 8.5),
                          gridspec_kw={'height_ratios': [1.1, 1, 0.85],
                                       'hspace': 0.50, 'wspace': 0.10})
fig.subplots_adjust(left=0.09, right=0.96, top=0.91, bottom=0.03)

fgsm_pert_np = fgsm_snaps[1]['image'] - orig_np
fgsm_adv_np  = fgsm_snaps[1]['image']

# ── Row 0: FGSM — "Original + Perturbation = Adversarial" ───
fgsm_items = [
    (0, orig_np,     'gray',   'Original',
     f'Pred: {fgsm_snaps[0]["pred"]} ({fgsm_snaps[0]["conf"]*100:.0f}%)', '#2166AC'),
    (2, fgsm_pert_np, 'RdBu_r', 'ε · sign(∇L)',
     'FGSM perturbation', '#888800'),
    (4, fgsm_adv_np, 'gray',   'Adversarial',
     f'Pred: {fgsm_snaps[1]["pred"]} ({fgsm_snaps[1]["conf"]*100:.0f}%)',
     '#D6001C' if fgsm_snaps[1]['pred'] != true_cls else '#2166AC'),
]
for col, data, cmap, title, xlabel, clr in fgsm_items:
    ax = axes[0, col]
    if cmap == 'RdBu_r':
        ax.imshow(data, cmap=cmap, vmin=-0.12, vmax=0.12, interpolation='nearest')
    else:
        ax.imshow(data, cmap=cmap, vmin=0, vmax=1, interpolation='nearest')
    for sp in ax.spines.values(): sp.set_edgecolor(clr); sp.set_linewidth(2.0)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=12, color=clr, fontweight='bold', pad=5)
    ax.set_xlabel(xlabel, fontsize=10, color=clr, labelpad=3)

# "+" and "=" symbols
for col, sym in [(1, '+'), (3, '=')]:
    ax = axes[0, col]
    ax.axis('off')
    ax.text(0.5, 0.5, sym, ha='center', va='center',
            fontsize=32, color='#555', fontweight='bold',
            transform=ax.transAxes)

# Col 5: FGSM explanation
ax = axes[0, 5]
ax.axis('off')
ax.text(0.5, 0.5,
        'FGSM:\n'
        'Compute gradient once,\n'
        'apply full ε perturbation\n'
        'in a single step.',
        ha='center', va='center', fontsize=10, color='#E08214',
        linespacing=1.5, transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.6', fc='#FFF8EE',
                  ec='#E08214', lw=1.0, alpha=0.92))

# Row label
axes[0, 0].set_ylabel('FGSM\n(1 step)', fontsize=12, color='#E08214',
                       fontweight='bold', labelpad=10)

# ── Separator line ───────────────────────────────────────────
ax_ov = fig.add_axes([0, 0, 1, 1], facecolor='none')
ax_ov.set_xlim(0, 1); ax_ov.set_ylim(0, 1); ax_ov.axis('off')
y_sep = 0.635
ax_ov.plot([0.09, 0.96], [y_sep, y_sep], color='#CCCCCC',
           linewidth=1.5, linestyle='--', clip_on=False)
ax_ov.text(0.525, y_sep - 0.03,
           'PGD = iterating many small FGSM-like steps (step size α = ε/4)',
           ha='center', va='bottom', fontsize=10.5, color='#D6001C',
           style='italic',
           bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ddd', lw=0.6))

# ── Row 1: PGD images at key steps ───────────────────────────
for j, step in enumerate(key_steps):
    s  = pgd_snaps[step]
    ax = axes[1, j]
    ax.imshow(s['image'], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    clr = '#D6001C' if s['pred'] != true_cls else '#2166AC'
    for sp in ax.spines.values(): sp.set_edgecolor(clr); sp.set_linewidth(1.8)
    ax.set_xticks([]); ax.set_yticks([])
    lbl = 'Init' if step == 0 else f'Step {step}'
    ax.set_title(lbl, fontsize=11, color='#333', fontweight='bold', pad=4)
    ax.set_xlabel(f'Pred: {s["pred"]} ({s["conf"]*100:.0f}%)',
                  fontsize=10, color=clr, fontweight='bold', labelpad=3)
    if j < len(key_steps) - 1:
        ax.annotate('', xy=(1.22, 0.5), xytext=(1.05, 0.5),
                    xycoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', lw=1.3, color='#999'))

axes[1, 0].set_ylabel('PGD\n(40 steps)', fontsize=12, color='#D6001C',
                       fontweight='bold', labelpad=10)

# ── Row 2: PGD perturbation at each step ─────────────────────
for j, step in enumerate(key_steps):
    s   = pgd_snaps[step]
    ax  = axes[2, j]
    pert = s['image'] - orig_np
    ax.imshow(pert, cmap='RdBu_r', vmin=-0.12, vmax=0.12, interpolation='nearest')
    for sp in ax.spines.values(): sp.set_edgecolor('#888'); sp.set_linewidth(1.0)
    ax.set_xticks([]); ax.set_yticks([])
    linf = np.abs(pert).max()
    ax.set_xlabel(f'L∞ = {linf:.3f}', fontsize=9, color='#555', labelpad=2)

axes[2, 0].set_ylabel('Perturbation\n(current − original)', fontsize=10,
                       color='#888', fontweight='bold', labelpad=10)

ax_ov.text(0.525, 0.225,
    'Red = pixel increased  |  Blue = pixel decreased  |  L∞ = max pixel change (capped at ε = 0.1)',
    ha='center', va='bottom', fontsize=9.5, color='#666',
    bbox=dict(boxstyle='round,pad=0.35', fc='#F8F8F8',
              ec='#CCC', lw=0.6))

fig.suptitle(
    f'FGSM vs PGD: Step-by-Step Breakdown  (ε = 0.1, True label = {true_cls})',
    fontsize=15, y=0.97, fontweight='bold',
)
plt.show()

In [ ]:
robust_model = SimpleCNN().to(device)
robust_opt   = torch.optim.Adam(robust_model.parameters(), lr=0.001)

epoch_losses = []
for epoch in range(5):
    robust_model.train()
    running_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)

        # Generate adversarial examples on the fly using PGD
        # (7 steps is a fast but effective training-time approximation)
        adv = pgd_attack(robust_model, imgs, lbls,
                         epsilon=0.1, alpha=0.025, iters=7)

        # Train on the adversarial examples, not the clean ones
        robust_opt.zero_grad()
        loss = criterion(robust_model(adv), lbls)
        loss.backward()
        robust_opt.step()
        running_loss += loss.item()

    avg = running_loss / len(train_loader)
    epoch_losses.append(avg)
    print(f'Epoch {epoch+1}/5  |  avg loss: {avg:.4f}')

print('Adversarial training complete.')

# ══════════════════════════════════════════════════════════════
# Visualization: Adversarial Training Loss Curve
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(6.5, 4))

epochs_x = list(range(1, len(epoch_losses) + 1))

ax.plot(epochs_x, epoch_losses,
        color='#D6001C', linewidth=2.2, marker='o',
        markersize=7, markeredgecolor='#D6001C',
        markerfacecolor='white', markeredgewidth=2.0,
        zorder=5)

# Fill area under curve
ax.fill_between(epochs_x, epoch_losses,
                alpha=0.08, color='#D6001C', zorder=1)

# Annotate start and end
ax.annotate(f'{epoch_losses[0]:.3f}',
            xy=(1, epoch_losses[0]),
            xytext=(1.45, epoch_losses[0] + 0.04),
            fontsize=10, color='#D6001C',
            arrowprops=dict(arrowstyle='->', color='#D6001C',
                            lw=1.3, connectionstyle='arc3,rad=-0.2'),
            bbox=dict(boxstyle='round,pad=0.25', fc='white',
                      ec='#D6001C', lw=1.0, alpha=0.92))

ax.annotate(f'{epoch_losses[-1]:.3f}',
            xy=(len(epoch_losses), epoch_losses[-1]),
            xytext=(len(epoch_losses) - 0.5, epoch_losses[-1] + 0.06),
            fontsize=10, color='#D6001C',
            arrowprops=dict(arrowstyle='->', color='#D6001C',
                            lw=1.3, connectionstyle='arc3,rad=0.2'),
            bbox=dict(boxstyle='round,pad=0.25', fc='white',
                      ec='#D6001C', lw=1.0, alpha=0.92))

# Improvement annotation
drop_pct = (1 - epoch_losses[-1] / epoch_losses[0]) * 100
mid_y = (epoch_losses[0] + epoch_losses[-1]) / 2
ax_ov = fig.add_axes([0, 0, 1, 1], facecolor='none')
ax_ov.axis('off')
# right-side note
ax_ov.text(0.82, 0.55,
           f'Loss dropped\n{drop_pct:.0f}%',
           transform=ax_ov.transAxes,
           fontsize=11, color='#666666', style='italic',
           ha='center', va='center',
           bbox=dict(boxstyle='round,pad=0.4', fc='#FFF5F5',
                     ec='#D6001C', lw=0.8, alpha=0.8))

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Average Loss', fontsize=12)
ax.set_title('Adversarial Training Loss  (PGD-7, $\\varepsilon$=0.1)',
             fontsize=13, pad=10)
ax.set_xticks(epochs_x)
ax.set_xlim(0.6, len(epoch_losses) + 0.4)

ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

plt.tight_layout()
plt.show()


In [ ]:
def evaluate_both(m, imgs, lbls, pgd_eps=0.1):
    """Returns (clean_acc, pgd_acc) for a model on the given batch."""
    m.eval()
    with torch.no_grad():
        clean = (m(imgs).argmax(1) == lbls).float().mean().item()
    pgd_adv = pgd_attack(m, imgs, lbls, epsilon=pgd_eps, iters=40)
    with torch.no_grad():
        pgd   = (m(pgd_adv).argmax(1) == lbls).float().mean().item()
    return clean, pgd

imgs, lbls = next(iter(test_loader))
imgs, lbls = imgs[:256].to(device), lbls[:256].to(device)

std_clean, std_pgd  = evaluate_both(model,        imgs, lbls)
rob_clean, rob_pgd  = evaluate_both(robust_model, imgs, lbls)

print(f'{"Model":<26} {"Clean acc":>12} {"PGD-40 acc":>12}')
print('-' * 52)
print(f'{"Standard training":<26} {std_clean*100:>11.1f}% {std_pgd*100:>11.1f}%')
print(f'{"Adversarial training":<26} {rob_clean*100:>11.1f}% {rob_pgd*100:>11.1f}%')

# ══════════════════════════════════════════════════════════════
# Visualization: Standard vs Adversarial Training Comparison
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(6.5, 4.5))

labels = ['Clean Accuracy', 'PGD-40 Accuracy']
std_vals = [std_clean * 100, std_pgd * 100]
rob_vals = [rob_clean * 100, rob_pgd * 100]

x = np.arange(len(labels))
width = 0.32

bars1 = ax.bar(x - width/2, std_vals, width, label='Standard Training',
               color='#2166AC', edgecolor='white', linewidth=0.8, zorder=3)
bars2 = ax.bar(x + width/2, rob_vals, width, label='Adversarial Training',
               color='#D6001C', edgecolor='white', linewidth=0.8, zorder=3)

# Value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1.2,
                f'{h:.1f}%', ha='center', va='bottom',
                fontsize=10.5, fontweight='bold',
                color=bar.get_facecolor())

# Highlight the key result: adversarial training dramatically improves PGD robustness
if rob_pgd > std_pgd + 0.05:
    improvement = rob_pgd * 100 - std_pgd * 100
    ax_ov = fig.add_axes([0, 0, 1, 1], facecolor='none')
    ax_ov.axis('off')

    # Arrow from standard PGD bar to robust PGD bar
    ax_ov.annotate(
        f'+{improvement:.0f}pp',
        xy=(0.67, 0.42), xytext=(0.67, 0.25),
        xycoords='axes fraction', textcoords='axes fraction',
        transform=ax.transAxes,
        fontsize=12, fontweight='bold', color='#D6001C',
        ha='center', va='bottom',
        arrowprops=dict(arrowstyle='->', color='#D6001C',
                        lw=2.0, connectionstyle='arc3,rad=0'))

# Dashed reference line at 100%
ax.axhline(y=100, color='#999999', linewidth=0.8, linestyle='--', alpha=0.5, zorder=1)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Standard vs. Adversarial Training', fontsize=13, pad=10)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11.5)
ax.set_ylim(0, 115)
ax.legend(fontsize=10, frameon=True, edgecolor='#cccccc',
          fancybox=False, loc='upper right')

ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

# Light grid
ax.yaxis.grid(True, alpha=0.25, linestyle='-', zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


In [ ]:
def bit_depth_reduction(x, bits=4):
    """
    Quantize pixel values to 2^bits discrete levels.
    bits=4  -> 16 levels (less aggressive)
    bits=2  -> 4 levels  (more aggressive)
    Any perturbation smaller than 1/(2^bits - 1) gets rounded away.
    """
    levels = 2 ** bits - 1
    return torch.round(x * levels) / levels

def spatial_smoothing(x, k=3):
    """
    Replace each pixel with the average of its k x k neighborhood.
    k=3 -> 3x3 average (mild blur)
    k=5 -> 5x5 average (stronger blur)
    """
    channels = x.shape[1]
    kernel   = torch.ones(1, 1, k, k, device=x.device) / (k * k)
    kernel_c = kernel.expand(channels, 1, -1, -1)
    return F.conv2d(x, kernel_c, padding=k//2, groups=channels)

# Get a batch and generate adversarial examples
imgs, lbls   = next(iter(test_loader))
imgs, lbls   = imgs[:256].to(device), lbls[:256].to(device)
adv_imgs     = pgd_attack(model, imgs, lbls, epsilon=0.1)

def acc_on(x):
    model.eval()
    with torch.no_grad():
        return (model(x).argmax(1) == lbls).float().mean().item()

results = {
    'Clean (no attack)':              acc_on(imgs),
    'PGD-40 (no defense)':            acc_on(adv_imgs),
    'Bit-depth 4-bit + PGD-40':       acc_on(bit_depth_reduction(adv_imgs, 4)),
    'Bit-depth 2-bit + PGD-40':       acc_on(bit_depth_reduction(adv_imgs, 2)),
    'Spatial smooth 3×3 + PGD-40':    acc_on(spatial_smoothing(adv_imgs, 3)),
    'Spatial smooth 5×5 + PGD-40':    acc_on(spatial_smoothing(adv_imgs, 5)),
}

print(f'{"Condition":<35} {"Accuracy":>10}')
print('-' * 48)
for name, val in results.items():
    bar = chr(9608) * int(val * 30)
    print(f'{name:<35} {val*100:>8.1f}%  {bar}')

# ══════════════════════════════════════════════════════════════
# Visualization 1: Defense Effectiveness (Horizontal Bar Chart)
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(7.5, 4))

names = list(results.keys())
vals  = [v * 100 for v in results.values()]

# Color coding: clean=blue, no-defense=dark red, bit-depth=amber, smooth=teal
bar_colors = ['#2166AC', '#D6001C', '#E08214', '#C06A10', '#2A9D8F', '#1A7A6E']

y_pos = np.arange(len(names))[::-1]  # top-to-bottom order

bars = ax.barh(y_pos, vals, height=0.6, color=bar_colors,
               edgecolor='white', linewidth=0.8, zorder=3)

# Value labels
for bar, v in zip(bars, vals):
    xpos = v + 1.2
    if v > 80:
        xpos = v - 5
        ax.text(xpos, bar.get_y() + bar.get_height()/2,
                f'{v:.1f}%', va='center', ha='right',
                fontsize=10.5, fontweight='bold', color='white')
    else:
        ax.text(xpos, bar.get_y() + bar.get_height()/2,
                f'{v:.1f}%', va='center', ha='left',
                fontsize=10.5, fontweight='bold',
                color=bar.get_facecolor())

ax.set_yticks(y_pos)
ax.set_yticklabels(names, fontsize=10.5)
ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_title('Input Transformation Defense Effectiveness', fontsize=13, pad=10)
ax.set_xlim(0, 108)

# Reference lines
ax.axvline(x=vals[0], color='#2166AC', linewidth=0.9, linestyle='--',
           alpha=0.4, zorder=1, label='Clean baseline')
ax.axvline(x=vals[1], color='#D6001C', linewidth=0.9, linestyle='--',
           alpha=0.4, zorder=1, label='No defense')

ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

ax.xaxis.grid(True, alpha=0.2, linestyle='-', zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# Visualization 2: Visual Effect of Input Transformations
# ══════════════════════════════════════════════════════════════

# Pick a sample adversarial image
sample_idx = 0
for i in range(len(adv_imgs)):
    pred = model(adv_imgs[i:i+1]).argmax(1).item()
    true = lbls[i].item()
    if pred != true and true in [4, 7, 8, 9, 5]:
        sample_idx = i
        break

sample_adv  = adv_imgs[sample_idx:sample_idx+1]
sample_clean = imgs[sample_idx:sample_idx+1]
true_label  = lbls[sample_idx].item()

transforms_list = [
    ('Clean\nOriginal',     sample_clean,                                '#2166AC'),
    ('Adversarial\n(PGD-40)',   sample_adv,                              '#D6001C'),
    ('Bit-depth\n4-bit',    bit_depth_reduction(sample_adv, 4),          '#E08214'),
    ('Bit-depth\n2-bit',    bit_depth_reduction(sample_adv, 2),          '#C06A10'),
    ('Smooth\n3×3',         spatial_smoothing(sample_adv, 3),            '#2A9D8F'),
    ('Smooth\n5×5',         spatial_smoothing(sample_adv, 5),            '#1A7A6E'),
]

fig, axes = plt.subplots(1, 6, figsize=(10, 2.2))

for ax_i, (title, tensor, clr) in zip(axes, transforms_list):
    img_np = tensor[0, 0].detach().cpu().numpy()

    # Get prediction
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        pred = logits.argmax(1).item()
        conf = torch.softmax(logits, 1).max().item()

    ax_i.imshow(img_np, cmap='gray', vmin=0, vmax=1)
    ax_i.set_title(title, fontsize=9.5, color=clr, fontweight='bold', pad=6)
    ax_i.set_xticks([])
    ax_i.set_yticks([])

    # Colored border
    for spine in ax_i.spines.values():
        spine.set_edgecolor(clr)
        spine.set_linewidth(2.5)

    # Prediction label below
    pred_color = '#2166AC' if pred == true_label else '#D6001C'
    ax_i.set_xlabel(f'pred: {pred}  ({conf*100:.0f}%)',
                    fontsize=8.5, color=pred_color, labelpad=3)

fig.suptitle(f'Effect of Input Transformations on Adversarial Example  (true label: {true_label})',
             fontsize=12, y=1.05)

plt.tight_layout()
plt.show()
